# E9.5 ATAC preprocessing: peaks, coverage, hotspot classes

Prepares the ATAC data of the E9.5 Dnmt-DKO chimera head parts (host and DKO) for the Fig. 6 analyses. No paper panels.

Inputs (misha tracks):
- `embdnmt.brain_e95_{host,dko}.marginal` (`prep/E9.5_marginal_tracks.ipynb`).
- `embdnmt.brain_e95_{host,dko}.reannot.<state>` and `embdnmt.brain_e95_dko.marginal_plus_gastr` (`prep/E9.5_atac_tracks.ipynb`).

Outputs (via `out_path()`, in `revision_e95/data/output/`):
- `npeaks.RDS`: E9.5 peak set.
- `multi_type_cov.RDS`, `cov_peak_chr.RDS`, `cov_chr.RDS`: coverage per state; read by `init_e95` (`E9.5_multiome.Fig6/Fig6_func.r`).
- `ehs.RDS`: E9.5 hotspot classes (see the caveat in that section).

Cells that do not run as stored are marked.

## Setup

Sources `paths.R` and `scripts/ATAC_preprocessing.r`, then switches to `embdnmt_dir`, so relative paths resolve against the original analysis directory.

In [ ]:
source("../paths.R")

In [1]:
library(dplyr)
library(tidyr)
library(ggplot2)

In [2]:
setwd(embdnmt_dir)

source(file.path(repo_dir, "scripts/ATAC_preprocessing.r"))
source(file.path(repo_dir, "E9.5_multiome.Fig6/e95_func.r"))
options(max.data.size = 1e+15)

## Regional normalization track (`rewrite_tracks`)

`comp_capped_regional_atac` makes `<track>_regcap`: 300 bp coverage, capped at the 95th percentile, averaged over 20 kb. It is the peak caller's denominator.
- Written to the shared misha database (`mm10_trackdb`), not `git_test/`.
- The tracks exist and `gtrack.smooth` cannot overwrite, so `rewrite_tracks = F` only reloads the database. To rebuild, first remove them (the disabled `if (0)` block); other notebooks read these tracks.
- `marginal_trs` names only `embdnmt.brain_e95_dko.marginal_plus_gastr`, whose `_regcap` is used by `init_e95` for the Fig. 6 CRE peaks (`peaks_on_sum.RDS`).
- The host/DKO `_regcap` tracks used below match the commented-out `marginal_trs`.

In [ ]:
# Set to TRUE to (re)write the regionally capped marginal track (<track>_regcap) into the shared misha track database.
# The tracks already exist, so the default only reads them.
rewrite_tracks = F

In [3]:
marginal_trs = "embdnmt.brain_e95_dko.marginal_plus_gastr"

if (0){
    for (tr in paste0(marginal_trs, "_regcap")){
        gtrack.rm(tr, force = T)
    }
}

gdb.reload()

if (rewrite_tracks) {
    for (tr in marginal_trs) {
        comp_capped_regional_atac(tr)
        message(paste(tr, "done."))
    }
}

gdb.reload()

## Peak calling

`join_norm_peaks.e95` (`E9.5_multiome.Fig6/e95_func.r`) calls peaks on host and DKO separately and merges them.
- Ratio per 20 bp bin: 300 bp coverage / regional `_regcap` coverage (`eps = 3`).
- Keeps bins above the 97th percentile (`T_q = 0.97`).
- Writes `npeaks.RDS`. Used only for the coverage statistics; the Fig. 6 CREs are called in `init_e95`.

In [ ]:
peaks = join_norm_peaks.e95()

In [ ]:
saveRDS(peaks, out_path("./revision_e95/data/output/npeaks.RDS"))

## Coverage statistics per cell state

`collect_cov_r_npeaks.e95` (`E9.5_multiome.Fig6/e95_func.r`; as `collect_cov_r.e95`, but reads `npeaks.RDS` itself) records per `reannot.<state>` track:
- `tot`: total coverage.
- `tot_peak`: coverage on the E9.5 peaks.
- `tot_rp`: coverage within 1 kb of ribosomal-protein TSSs (`Rpl*`, `Rps*`), the housekeeping standard for host vs DKO.
- Coverage per chromosome, for the trisomy correction.

Writes `multi_type_cov.RDS`, `cov_peak_chr.RDS`, `cov_chr.RDS`.

In [ ]:
collect_cov_r_npeaks.e95()

## TSS intervals

Loads mm10 TSSs (`intervs.global.tss`) without chrM and chrY.

In [55]:
tss = gintervals.load("intervs.global.tss")
tss = tss[tss$chrom != "chrM" & tss$chrom != "chrY",]

## Karyotype and coverage normalization

Reloads the outputs above and an `ehs.RDS` from an earlier run of the classification below.
- `r_chr_cov_dko`: per state, DKO/host coverage per chromosome over its median; > 1.06 in `marginal` marks candidate trisomies (Extended Data Fig. 10h, `prep/E9.5_marginal_tracks.ipynb`).
- `cov_stat_kary_norm`: DKO totals divided by the karyotype inflation (`dko.tot`, `dko.tot_peak`, `dko.tot_rp`).
- `states`: cell states with coverage in both genotypes.

Caveats:
- It does not correct the chr14 trisomy; the scatters drop chr14 instead, and the Fig. 6 peaks have no chr14 peaks.
- `tr_karyo` and `r_to_dsamp` are not used further.

In [3]:
peaks = readRDS(in_path("./revision_e95/data/output/npeaks.RDS"))
cov_chr = readRDS(in_path("./revision_e95/data/output/cov_chr.RDS"))
cov_stat = readRDS(in_path("./revision_e95/data/output/multi_type_cov.RDS"))
ehs = readRDS(in_path("./revision_e95/data/output/ehs.RDS"))

In [4]:
r_chr_cov_dko = cov_chr[["brain_e95_dko.reannot"]]/cov_chr[["brain_e95_host.reannot"]]
r_chr_cov_dko = r_chr_cov_dko/apply(r_chr_cov_dko, 1, median, na.rm=T)

which(r_chr_cov_dko["marginal",] > 1.06)

tr_karyo = list()
tr_karyo[["host"]] = NULL
tr_karyo[["dko"]] = r_chr_cov_dko["marginal",]


tot_host = gsummary("embdnmt.brain_e95_host.reannot.marginal")[["Sum"]]
tot_dko = gsummary("embdnmt.brain_e95_dko.reannot.marginal")[["Sum"]]

min_tot = pmin(tot_host, tot_dko)
r_to_dsamp = data.frame(host = min_tot/tot_host, 
                        dko = min_tot/tot_dko)

In [6]:
r_chr_cov_dko = cov_chr[["brain_e95_dko.reannot"]]/cov_chr[["brain_e95_host.reannot"]]
r_chr_cov_dko = r_chr_cov_dko/apply(r_chr_cov_dko, 1, median, na.rm=T)

dko_tot_karyo = rowSums(cov_chr[["brain_e95_dko.reannot"]],na.rm=T)/rowSums(cov_chr[["brain_e95_dko.reannot"]]/r_chr_cov_dko, na.rm=T)
	
cov_norm = cov_stat
cov_norm$dko.tot = floor(cov_stat$brain_e95_dko.reannot.tot / dko_tot_karyo)
cov_norm$dko.tot_peak = floor(cov_stat$brain_e95_dko.reannot.tot_peak / dko_tot_karyo)
cov_norm$dko.tot_rp = floor(as.numeric(cov_stat$brain_e95_dko.reannot.tot_rp) / dko_tot_karyo)

cov_stat_kary_norm = cov_norm
rownames(cov_stat_kary_norm) = cov_stat_kary_norm$nm

In [8]:
states = rownames(cov_chr[[1]])

In [9]:
states1 = cov_chr[[1]][,1]
states1 = names(states1[!is.na(states1)])
states2 = cov_chr[[2]][,1]
states2 = names(states2[!is.na(states2)])

states = intersect(states1, states2)

## CpG and GC content of the peaks

Adds CpG content, GC fraction and nearest TSS to `npeaks`.
- The second cell fails in a fresh session: it bins columns of `peaks`, but they were added to `npeaks`. Its outputs are used only by the pasted plotting code below.

In [53]:
source(file.path(repo_dir, "scripts/lineage_peaks_func.r"))
gvtrack.create("GC500", "seq.G_or_C", "sum")
npeaks = peaks
all_cg = gextract(c("seq.CG_500_mean_new", "GC500"), iterator=peaks,
                            intervals=npeaks, colnames=c("CG", "GC"))
all_cg$GC = all_cg$GC/(all_cg$end - all_cg$start)
npeaks$CG = all_cg$CG
npeaks$GC = all_cg$GC

In [56]:
a = gintervals.neighbors(npeaks, tss[,c("chrom", "start", "end", "strand", "geneSymbol")])
npeaks$d_tss = a$dist
npeaks$gene_tss  = a$geneSymbol
npeaks$gene_strand  = a$strand

cgbin = as.numeric(cut(peaks$CG, c(seq(0, 0.05, 0.01), 
    1), include.lowest = T))
gcbin = as.numeric(cut(peaks$GC, c(0, seq(0.4, 0.6, 
    0.05), include.lowest = T)))
f_tss = abs(peaks$d_tss) < 1000 | abs(peaks$d_tss) > 
    1e+06
f_hgc = peaks$GC > 0.55
f_lcg = peaks$CG < 0.015

## E9.5 hotspot classes (does not run as stored)

The `build_epi_hotspots_classes` partition applied to the E9.5 peaks: classes by CTCF strength, TSS distance (1 kb), max CpG content (0.04) and ATAC overlap. Enhancers are not split by accessibility.
- Fails in a fresh session: `ctcf_peak` and `cgweak_intervs` are not defined here.
- Writes `out_path("./data/output/ehs.RDS")`, i.e. the gastrulation location, not `revision_e95/data/output/ehs.RDS`, which the normalization section reads.

In [ ]:
u1 = gintervals.union(intervals1=npeaks, intervals2=ctcf_peak)
u2 = gintervals.union(intervals1=u1, intervals2 = tss)
ehs = gintervals.union(intervals1=u2, intervals2 = cgweak_intervs)

ehs = ehs[ehs$chrom != "chrM" & ehs$chrom != "chrY",]

gvtrack.create("cg_max", "seq.CG_500_mean_new", "max")
e_cg = gextract(c("cg_max"), intervals=ehs, iterator=ehs, colnames=c("cg_max"))
ehs$cg_max = e_cg$cg_max

ehs_peak = gintervals.neighbors1(ehs, npeaks, na.if.notfound=T)
ehs_tss = gintervals.neighbors1(ehs, tss, na.if.notfound=T)
ehs_cgweak = gintervals.neighbors1(ehs, cgweak_intervs, na.if.notfound=T)
ehs_ctcf = gintervals.neighbors1(ehs, ctcf_peak, na.if.notfound=T)

ehs$atac_d = ehs_peak$dist

ehs$ctcf_d = ehs_ctcf$dist
ehs$ctcf_chip = ehs_ctcf$chip
ehs$ctcf_e = ehs_ctcf$e_CTCF
ehs$ctcf_strand = ehs_ctcf$strand

ehs$tss_d = ehs_tss$dist
ehs$tss_strand = ehs_tss$strand

ehs$cg_d = ehs_cgweak$dist
ehs$cg_CG = ehs_cgweak$CG
ehs$cg_GC = ehs_cgweak$GC

#partition all hotspts:
ehs$mode = "none";
ehs$center = floor((ehs$start+ehs$end)/2)
ehs$strand = ehs$tss_strand

#CTCF non TSS
f_ins_sctcf = ehs$ctcf_d == 0 & ehs$ctcf_chip >= 50 & abs(ehs$tss_d) >= 1000
ehs$mode[f_ins_sctcf] = "ins_sctcf"
f_ins_wctcf = ehs$ctcf_d == 0 & ehs$ctcf_chip < 50 & abs(ehs$tss_d) >= 1000
ehs$mode[f_ins_wctcf] = "ins_wctcf"
f_ins_ctcf = f_ins_sctcf | f_ins_wctcf
ehs$strand[f_ins_ctcf] = ehs$ctcf_strand[f_ins_ctcf]
ehs$center[f_ins_ctcf] = ehs_ctcf$start1[f_ins_ctcf]

#TSS w CTCF (center on CTCF)
f_tss_ctcf = ehs$ctcf_d == 0 & abs(ehs$tss_d) < 1000
ehs$mode[f_tss_ctcf] = "tss_ctcf"
ehs$strand[f_tss_ctcf] = ehs$tss_strand[f_tss_ctcf]
ehs$center[f_tss_ctcf] = ehs_ctcf$start1[f_tss_ctcf]

#TSS w/o CTCF
f_tss = abs(ehs$tss_d) == 0 & ehs$mod == "none"
ehs$mode[f_tss & ehs$cg_d == 0 & ehs$cg_max>0.04] = "tss_hcg"
ehs$mode[f_tss & ehs$cg_d == 0 & ehs$cg_max<0.04] = "tss_icg"
ehs$mode[f_tss & ehs$cg_d != 0] = "tss_lcg"
ehs$strand[f_tss] = ehs$tss_strand[f_tss]
ehs$center[f_tss] = ehs_tss$start1[f_tss]

#atac high CG non TSS
ehs$mode[abs(ehs$cg_d) == 0 & ehs$cg_max > 0.04 & abs(ehs$atac_d) == 0 & ehs$mod == "none"] = "hcg_enh"
#atac mid CG non TSS
ehs$mode[abs(ehs$cg_d) == 0 & ehs$cg_max < 0.04 & abs(ehs$atac_d) == 0 & ehs$mod == "none"] = "icg_enh"
#atac low CG non TSS
ehs$mode[ehs$atac_d == 0 & ehs$mod == "none"] = "lcg_enh"

#atac high CG non TSS
ehs$mode[abs(ehs$cg_d) == 0 & ehs$cg_max > 0.04 & ehs$mod == "none"] = "hcg_void"
#atac mid CG non TSS
ehs$mode[abs(ehs$cg_d) == 0 & ehs$cg_max < 0.04 & ehs$mod == "none"] = "icg_void"
#atac low CG non TSS
ehs$mode[abs(ehs$cg_d) != 0 & ehs$cg_max < 0.04 & ehs$mod == "none"] = "lcg_void"

In [ ]:
saveRDS(ehs, out_path("./data/output/ehs.RDS"))